# Indexing

> Selecting and changing parts of arrays: positions, blocks, one item per row or column, sorted columns, coordinates, paths, windows, edges and named indices

BPL has three ways to select. `I⌷A` takes selectors for leading axes and gives every combination of them. `A.[I J]` pairs its arrays of positions item by item, as NumPy's advanced indexing does. `P⊃A` follows a path into nested arrays. Each section below shows an everyday task and the form that does it. [Axis keys](keyed.ipynb) covers selecting by key.

## Positions

Positions count from 0. A negative position counts from the end, as in Python:

In [ ]:
v←10 20 30 40 50
v.[[0 ¯1]]
v₋₁

10 50

50

Reading at a position outside its axis, or at one that is not a whole number, returns the fill. Writing there does nothing:

In [ ]:
v.[[7 1.5 1]]
v.[[7 1.5]]←99 99
v

0 0 20

10 20 30 40 50

[Reverse](glyphs/circle-stile.qmd) `⌽` reverses the last axis:

In [ ]:
m←[0 1 2 3◇4 5 6 7◇8 9 10 11]
⌽m

 3  2 1 0
 7  6 5 4
11 10 9 8

`:` selects one whole axis, and `∞` every axis that the other selectors leave:

In [ ]:
c←2 3 4ρι24
ρ[: 1]⌷c
ρ[∞ 1]⌷c

[2 4]ₓ

[2 3]ₓ

`¯∞` inserts an unnamed, unkeyed axis of length one without consuming an input axis. It works with both `⌷` and `.[]`, as Python's `None` does:

In [ ]:
ρ[¯∞ ∞]⌷m
ρm.[: ¯∞ :]

[1 3 4]ₓ

[3 1 4]ₓ

## Blocks of rows and columns

`⌷` selects every combination of its positions. Here are rows 1 and 2 with columns 0 and 3:

In [ ]:
[1 2;0 3]⌷m

4  7
8 11

`.[]` pairs its arrays instead. With the same two arrays, `.[]` reads the items of `m` at row 1, column 0 and at row 2, column 3:

In [ ]:
m.[[1 2] [0 3]]

4 11

Arrays of positions with different axis names don't pair. Each keeps its axis, so selecting with them gives a block, as in xarray's broadcasting by dimension name:

In [ ]:
x←["i":5 "j":5]ρι25
r←1ₓ+ι["i":3ₓ] ◇ c←2ₓ+ι["j":2ₓ]
x.[r c]

 7  8
12 13
17 18

## One item from each row or column

Use `.[]` to pick one item from each row. `k` holds a column number for each row of `m`, as in NumPy's `m[np.arange(3), k]`:

In [ ]:
k←[2 0 3]ₓ
m.[ι3ₓ k]

2 4 11

To compute a classifier's loss, read the probability of each sample's true class. Each column of `P` holds one sample's probabilities for three classes. `L` holds each sample's true class. The loss is the mean negative log of those probabilities:

In [ ]:
P←[0.7 0.1 0.2 0.1◇0.2 0.8 0.3 0.1◇0.1 0.1 0.5 0.8]
L←[0 1 2 2]ₓ
P.[L ι4ₓ]
- +/÷≢ ⊛P.[L ι4ₓ]

0.7 0.8 0.5 0.8

0.3740273068

The same pick works with named axes. `labels` holds one class number for each sample, along the sample axis `b`. Because `labels` and `b` share that axis name, their positions pair item by item. `Pn.[labels b]` uses the selectors in the order of `Pn`'s axes: `labels` gives the row and `b` gives the column, as in `P.[L ι4ₓ]`:

In [ ]:
Pn←["c":3 "b":4]ρP
labels←["b":4]ρL ◇ b←ι["b":4]
Pn.[labels b]

["b":4]ρ[0.7 0.8 0.5 0.8]

When the same array appears twice, `.[]` pairs it with itself, which gives a diagonal:

In [ ]:
i←ι3ₓ ◇ m.[i i]

0 5 10

## Sorting each column

`⍋⍠"i" sc` gives the row positions that sort each column of `sc`. `gr` has both of `sc`'s axis names. `sc.[gr]` uses `gr` to select rows, and pairs `gr`'s `j` axis with `sc`'s columns. Each column of `sc` is read in sorted order, as NumPy's `take_along_axis` does with `argsort`:

In [ ]:
sc←["i":3 "j":4]ρ[3 1 4 1◇5 9 2 6◇5 3 5 8]
gr←⍋⍠"i" sc
gr
sc.[gr]

[0 0 1 0
 1 2 0 1
 2 1 2 2]ₓ

3 1 2 1
5 3 4 6
5 9 5 8

## Coordinates

`⍸` gives the coordinates where a condition holds. Passing those coordinates to `.[]` reads the item at each one:

In [ ]:
m.[⍸m>8]

9 10 11

Assigning through the same selector changes those items:

In [ ]:
z←m ◇ z.[⍸z>8]←0 ◇ z

0 1 2 3
4 5 6 7
8 0 0 0

## Paths

`⊃` follows a path into nested arrays, one level for each item of the path. Here is item 2 of item 1:

In [ ]:
n←[1 2;3 4 5]
[1 2]⊃n

5

[Index](glyphs/squad.qmd) `⌷` also takes arrays of paths. Use `¯∞` to place new axes before the paths or among the selected cells' axes.

## Windows and neighbours

Adding two indices with different axis names builds a table of positions. Reading a vector through that table gives sliding windows:

In [ ]:
w←10 20 30 40 50 ◇ s←ι["s":3] ◇ t←ι["t":3]
s+t
w.[s+t]

0 1 2
1 2 3
2 3 4

10 20 30
20 30 40
30 40 50

To take the differences between neighbours, read the vector twice, one position apart, and subtract:

In [ ]:
y←10 20 40 70 ◇ p←ι["p":3]
y.[p+1]-y.[p]

["p":3]ρ[10 20 30]

## Edges

Arithmetic on positions can produce a position outside the axis. At the start, `e-1` is `¯1`, the position of the last item:

In [ ]:
u←10 20 30 40 ◇ e←ι4ₓ
u.[e-1ₓ]

40 10 20 30

Use [`•edge`](system/edge.qmd) to control what happens to positions outside the axis. `4•edge e-1` replaces each position outside an axis of length 4 with NaN. Reading at NaN returns the fill because NaN isn't a whole number, as described under [positions](#positions). Use [`⌷⍠["fill":v]`](glyphs/quad-colon.qmd#fill) to read `v` instead of the fill at those positions. Reading `u` through these positions shifts it right, with a zero at the start:

In [ ]:
4•edge e-1
u.[4•edge⁝e-1]

$n 0 1 2

0 10 20 30

`•edge "clamp"` gives a function that moves each position outside the axis to the nearest end instead. `"wrap"`, `"reflect"` and `"symmetric"` are the other modes:

In [ ]:
clamp←•edge "clamp"
u.[(4 clamp e-1)]

10 10 20 30

On an axis of length 6, here is where each mode sends the positions from ¯2 to 5:

In [ ]:
[wrap reflect symmetric]←•edge "wrap" "reflect" "symmetric"
pos←¯2ₓ+ι8ₓ
6 clamp pos
6 wrap pos
6 reflect pos
6 symmetric pos

[0 0 0 1 2 3 4 5]ₓ

[4 5 0 1 2 3 4 5]ₓ

[2 1 0 1 2 3 4 5]ₓ

[1 0 0 1 2 3 4 5]ₓ

## Changing a selection

Any selection can be an assignment target:

In [ ]:
a←10 20 30 40 50 ◇ a.[[1 3]]←0 ◇ a

10 0 30 0 50

To write a block, use `⌷` with a space before `←`. The space makes the whole run before it the target:

In [ ]:
bk←3 6ρ0 ◇ [0 2;1 4]⌷bk +←10 ◇ bk

0 10 0 0 10 0
0  0 0 0  0 0
0 10 0 0 10 0

Modified assignment applies its function once for each time a position appears. Repeated keyed positions work too. The [indexing rules](glyphs/squad.qmd) describe when a selection drops keys. Counting a histogram takes one line:

In [ ]:
h←5ρ0 ◇ h.[[1 3 1 1]]+←1 ◇ h

0 3 0 1 0

Writing at a position that isn't a whole number, such as NaN, does nothing. Passing bin numbers through `•edge` first therefore drops the values that fall outside the bins, here 7 and ¯1:

In [ ]:
g←5ρ0 ◇ d←0.5 1.5 1.7 4.2 7 ¯1
g.[5•edge⌊d]+←1 ◇ g

1 2 0 0 1

Writing does nothing wherever reading returns a fill. `5↑abc` reads two fills past the end of `abc`. Assigning to `5↑abc` changes only the three items of `abc`:

In [ ]:
abc←1 2 3 ◇ 5↑abc ←10 20 30 40 50 ◇ abc

10 20 30

Reading the first item of an empty string returns a fill. `@` can therefore change the first letter of each word, empty words included, with no special case. `1•c` converts to uppercase:

In [ ]:
1•c@↑¨ "hello" "" "world"

"Hello" "" "World"

## Under

`f@g` applies `f` to the selection that `g` makes, then writes the result back:

In [ ]:
10× @[1 3]u

10 200 30 400

After a function, `.[I]` selects from its result. On its own, `.[I]` is equivalent to `⊢.[I]`, a selector function that Under can use directly:

In [ ]:
10× @.[[1 3]] u

10 200 30 400

`⊢.name` reads a field in the same way, as `{ω.name}` does:

In [ ]:
orders←[["price":10 "qty":2] ["price":5 "qty":1]]
⊢.price¨ orders

10 5

## Named and fresh indices

A named index on its own selects along the axis with the same name, wherever the index appears in the selector. `sales.[mo cy]` reads months with `mo` and cities with `cy`. The result is `sales` itself:

In [ ]:
sales←["city":"NY" "LA";"month":"Jan" "Feb" "Mar"]:[10 20 30◇40 50 60]
cy←ι⍠"city" sales ◇ mo←ι⍠"month" sales
sales≡sales.[mo cy]

$t

[`•index`](system/index.qmd) gives an index along an axis with a fresh name. Selecting with two fresh indices gives a grid. Subtracting `q.[ci]` from `q.[ri]` gives the difference between every pair of items in `q`:

In [ ]:
q←10 20 40 ◇ [ri ci]←•index 3 3
q.[ri]-q.[ci]

 0 ¯10 ¯30
10   0 ¯20
30  20   0